# M06-02 — Cache y particionado

[← Anterior](02-lab-explain-dag.ipynb) · [Siguiente →](../M07-persistencia-datos/01-teoria.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Materializar un cache con una acción y ver cómo `repartition("order_month")` cambia el número de particiones.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M06-02-cache-particionado.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Un fact más largo

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

8 copias: suficiente para notar el cache en local, sin saturar el Codespace.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Varias particiones (> 1). Count esperado **15840** cuando lo lances.

**Por qué este paso.** 1980 × 8. Poco para un clúster; bastante para ver Storage en local.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, lit

spark = get_spark("novashop-m06")
base = spark.read.parquet(str(STAGING / "fact_lines"))
xl = base.withColumn("_copy", lit(-1))
for i in range(7):
    xl = xl.unionByName(base.withColumn("_copy", lit(i)))
print("particiones iniciales", xl.rdd.getNumPartitions())


### Paso 2 — Dos counts sin cache

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Cada acción relee el plan desde el Parquet + unions. Dos jobs de coste parecido.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Dos tiempos del mismo orden. Cobrable = 1127 × 8 = **9016**.

**Por qué este paso.** En local a veces es tan corto que el cronómetro no emociona: mira Jobs.


In [ ]:
import time

def timed_count(df, label):
    t0 = time.perf_counter()
    n = df.where(col("is_billable")).count()
    print(label, n, f"{time.perf_counter() - t0:.2f}s")

timed_count(xl, "1er count frío")
timed_count(xl, "2º count frío")


### Paso 3 — Cache materializado

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

cache() no llena Storage hasta una acción. Primero calientas; después lees memoria.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** En Spark UI → Storage aparece el DataFrame. El segundo tiempo **no empeora**.

**Por qué este paso.** La primera acción llena Storage; la segunda debería leer memoria.

Al terminar: `warm.unpersist()` (otra celda, con su Markdown).


In [ ]:
warm = xl.where(col("is_billable")).cache()
timed_count(warm, "calentamiento (materializa cache)")
timed_count(warm, "caliente")


### Paso 4 — Repartition por mes

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

repartition(12, order_month) hace shuffle hacia 12 particiones. Es preparación para escribir (M07), no una window.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `particiones 12`. Doce meses en el groupBy.

**Por qué este paso.** Si haces `repartition(col)` sin `n`, en 3.5 usas 200 particiones por defecto.


In [ ]:
by_month = warm.repartition(12, col("order_month"))
print("particiones", by_month.rdd.getNumPartitions())
by_month.groupBy("order_month").count().orderBy("order_month").show()


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

Ejecuta solo `.cache()` y mira Storage **antes** de cualquier count → vacío.
Luego un count → aparece. Escríbelo.


## Mejora — coalesce vs repartition

Pasa a 1 partición con `coalesce(1)` y con `repartition(1)`. ¿Cuál declara shuffle en el plan?

Si te atasca, el código está en la celda siguiente.


In [ ]:
`repartition(1)` siempre shufflea. `coalesce(1)` reduce sin shuffle amplio. Útil para un único fichero de entrega; malo como hábito de pipeline (M07).


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| Storage vacío tras cache() | No hubo acción | count() o show() |
| OOM | Demasiadas copias + cache | Quédate en 8; unpersist |
| 1980 o 200 particiones | repartition sin n | repartition(12, col("order_month")) |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M07 — teoría](../M07-persistencia-datos/01-teoria.ipynb).
